# Trino Prod — Starter Notebook

Template for querying production Trino via `kubectl port-forward`. Use this as a starting point for ad-hoc analytics on the Giga Meter and Giga Maps databases.

## Prerequisites

1. **Access granted** to the `uni-ooi-giga-aks-prd` cluster (request from Alajos).
2. **CLIs installed:** `az`, `kubectl`, `kubelogin` (Azure version — `brew install Azure/kubelogin/kubelogin`, *not* the generic OIDC one).
3. **Authenticated:**
   ```bash
   az login
   az account set -s <SUBSCRIPTION_ID>  # ask the Giga DevOps team
   az aks get-credentials --name uni-ooi-giga-aks-prd --resource-group RS-UNI-GIGA-AKS-PRD
   kubelogin convert-kubeconfig -l azurecli
   ```
4. **Port-forward running in a separate terminal:**
   ```bash
   kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd
   ```
   Keep that terminal open the whole time you're querying.

## Python deps

Run once in your venv:
```bash
pip install trino "trino[sqlalchemy]" pandas matplotlib jupyterlab
```

## ⚠️ This is production

- Always use `LIMIT` while exploring.
- Avoid `SELECT *` on large fact tables (`measurements` is huge).
- Heavy queries are visible to the team — be considerate.

## 1. Connection setup

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from trino.dbapi import connect

# --- Connection settings ---
TRINO_HOST = "localhost"
TRINO_PORT = 8080
TRINO_USER = "giga-trino"  # identity used for Trino-level access control
TRINO_CATALOG = "delta_lake"  # Giga Maps prod also available — run SHOW CATALOGS to see all
TRINO_SCHEMA = "default"

# DB-API connection — use for raw cursor-style queries
conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    catalog=TRINO_CATALOG,
    schema=TRINO_SCHEMA,
    http_scheme="http",
)

# SQLAlchemy engine — use with pandas.read_sql
engine = create_engine(
    f"trino://{TRINO_USER}@{TRINO_HOST}:{TRINO_PORT}/{TRINO_CATALOG}/{TRINO_SCHEMA}"
)

print("Connected. Catalogs visible:", pd.read_sql("SHOW CATALOGS", engine)["Catalog"].tolist())

## 2. Discover the catalog

Confirm schemas and tables before writing real queries.

In [ ]:
# Schemas in the current catalog
pd.read_sql(f"SHOW SCHEMAS FROM {TRINO_CATALOG}", engine)

In [ ]:
# Tables in public schema
pd.read_sql(f"SHOW TABLES FROM {TRINO_CATALOG}.default", engine)

In [ ]:
# Inspect the measurements table schema
pd.read_sql(f"DESCRIBE {TRINO_CATALOG}.public.measurements", engine)

## 3. Example query — top 10 measurements

In [ ]:
df_measurements = pd.read_sql(
    """
    SELECT *
    FROM measurements
    LIMIT 10
    """,
    engine,
)
df_measurements

## 4. Helper — run any query

Use this for exploratory queries. Auto-injects `LIMIT 1000` if you forget it — safety net for prod.

In [ ]:
def q(sql: str, limit: int | None = 1000) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame. Auto-applies LIMIT if not present."""
    sql = sql.strip().rstrip(";")
    if limit is not None and "limit" not in sql.lower():
        sql = f"{sql}\nLIMIT {limit}"
    return pd.read_sql(sql, engine)


# Example: row count of measurements
q("SELECT COUNT(*) AS row_count FROM measurements", limit=None)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `Connection refused` on port 8080 | Port-forward terminal died. Re-run `kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd`. |
| `Forbidden` from kubectl | RBAC not applied yet — ping Alajos. |
| `PERMISSION_DENIED` from Trino | Check `TRINO_USER` matches what Trino's access control expects (currently `giga-trino`). |
| Query hangs forever | Laptop slept and broke the tunnel. Restart port-forward. |
| `kubelogin not found` | `brew install Azure/kubelogin/kubelogin`, then `kubelogin convert-kubeconfig -l azurecli`. |
| Auth expired after a day | `az login` again. |
| Need to query Giga Maps instead | Change `TRINO_CATALOG` to the maps prod db catalog name (run `SHOW CATALOGS` to see). |

One-time setup                                                                                           
                                                                                                           
  1. Get cluster access — request from Alajos (cluster: uni-ooi-giga-aks-prd).                             
  2. Install CLIs:                                                                                         
  brew install azure-cli kubectl                                                                           
  brew install Azure/kubelogin/kubelogin   # ⚠️  Azure version, not the generic OIDC one                  
  3. Authenticate:                                                                                         
  az login                                                                                               
  az account set -s <SUBSCRIPTION_ID>  # ask the Giga DevOps team                                                   
  az aks get-credentials --name uni-ooi-giga-aks-prd --resource-group RS-UNI-GIGA-AKS-PRD                  
  kubelogin convert-kubeconfig -l azurecli
                                                                                                           
  Each session                                                                                           
                                                                                                           
  Open a separate terminal and run:                                                                        
  kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd
  Leave that terminal open for as long as you're querying. If your laptop sleeps the tunnel breaks —       
  restart the command.                                                                                   
                                                                                                           
  Verify
                                                                                                           
  In another terminal:                                                                                   
  curl http://localhost:8080/v1/info
  Should return JSON. If you get Connection refused, the port-forward died — restart it.
           